In [25]:
import os
import polars as pl

In [26]:
script_path = os.getcwd()
project_path = os.path.join(script_path, '..', '..')
features_dir = os.path.join(project_path, 'data', 'features')
feature_file_path = os.path.join(features_dir, 'content_relevance_score.parquet')
feature_df = pl.read_parquet(feature_file_path)

In [27]:
# 1. Mostrar el contenido completo de los strings (sin límite)
pl.Config.set_fmt_str_lengths(1000) # Ajusta al largo que necesites

polars.config.Config

197_000, $ 

In [28]:
feature_df

comment_id,content_relevance_score,reasoning_content_relevance_score
str,i64,str
"""mzlqxrg""",4,"""The comment expresses a desire for Iran to take military action against Israel in the context of humanitarian aid, which relates directly to the ongoing conflict and its implications for Gaza."""
"""obbcoq3""",0,"""The comment is vague and does not provide any specific information or opinion related to the Gaza conflict, focusing instead on a general statement about truth without context."""
"""n9yjtzg""",2,"""The comment reflects on the hostility faced by Jews in Barcelona, linking it to broader themes of antisemitism and public sentiment related to the Gaza conflict, but it does not directly address the conflict itself or its core events."""
"""nax2wzf""",1,"""The comment focuses on a political strategy regarding the Democratic Party in the context of the Gaza conflict, but it does not directly address the humanitarian situation or the core events of the conflict itself. It is more about U.S. internal politics than the Gaza conflict."""
"""n8vxcji""",3,"""The comment reflects on a historical event related to Israel's actions in Gaza and connects it to current events, indicating a perspective on the conflict's implications. However, it does not engage with the core conflict or humanitarian issues directly."""
…,…,…
"""n10aq2v""",5,"""The comment explicitly discusses the ongoing violence and humanitarian implications of the conflict, focusing on the actions of Israel and Hamas, which are central to the Gaza conflict."""
"""mwt71mo""",5,"""The comment discusses the situation in Gaza, including the implications of journalist access and the actions of Hamas, which are directly related to the core conflict and the humanitarian situation in the region."""
"""nk1nmy7""",1,"""The comment primarily focuses on a specific individual accused of hate crimes and expresses disdain for public support of him, but it does not directly address the Gaza conflict or its core issues. Instead, it veers into personal attacks and generalizations about public opinion, which are not relevant to the conflict itself."""


In [18]:
count_relevant_content = sum(feature_df['content_relevance_score'] >= 3) 
content_size = len(feature_df)
prop_relevant_content = count_relevant_content / content_size
print(f"Count relevant content: {count_relevant_content}")
print(f"Percentage relevant content: {round(prop_relevant_content * 100, 2)}%")

Count relevant content: 97269
Percentage relevant content: 49.08%


In [19]:
feature_df['content_relevance_score'].value_counts().sort('count', descending=True)

content_relevance_score,count
i64,u32
4,46014
0,44584
2,33395
5,26572
3,24683
1,22946
-1,6


In [20]:
processed_data_dir = os.path.join(project_path, 'data', 'processed_data')

base_data_path = os.path.join(processed_data_dir, '02_processed_data.parquet') 

base_df = pl.read_parquet(base_data_path)

processed_df = base_df.join(feature_df, how='left', on='comment_id')

processed_df = processed_df[['comment_id', 'post_id', 'post_title', 'post_body', 'comment_body', 'content_relevance_score', 'reasoning_content_relevance_score']]

In [24]:
len(processed_df) - processed_df['content_relevance_score'].is_null().sum()

198200

In [35]:
for score in processed_df['content_relevance_score'].unique():
    if score:
        print(f'content_relevance_score == {score}'.upper())
        display(processed_df.filter(pl.col('content_relevance_score') == score).sample(n=5, seed=123))
        print('='*200, '\n')

CONTENT_RELEVANCE_SCORE == -1


comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""n8mj7ql""","""1mpr8o4""","""a video of aljazera journalist Anas, who was killed by Israel, talking to his daughter Sham.""","""""","""**Looks like this thread is getting a lot of attention. Greetings, /r/all! Please keep it civil.** *I am a bot, and this action was performed automatically. Please [contact the moderators of this subreddit](/message/compose/?to=/r/Palestine) if you have any questions or concerns.*""",-1,"""The comment is meta-Reddit moderation/attention ("""
"""npf2aoc""","""1ozxpc4""","""At least on Reddit, pro-Palestine subs do not say/discuss 'Globalize the Intifada' much. But pro-Israel subs absolutely do & obsess about it. The 'controversy' over this expression is exaggerated & weaponized to slander critics of Israel as antisemitic.""","""""","""Something, something, projection is the word that is appropriate here.""",-1,"""The comment is a terse, generic rebuttal ("""
"""ocakh06""","""1s2nzvc""","""I hope they realise the truth that they are supporting a gncidl entity.""","""Infront of Elstree and Borehamwood station, London, UK.""","""Oh they realise""",-1,"""Primary subject: accusation of supporting a 'genocidal entity' (core Gaza/Israel conflict) at a London location; the comment ("""
"""n7xwa20""","""1mm4i9x""","""If Pro-Palestinians think ""Zionism"" is an ideology rather than a reference to Jews, then why do they call all Jews who moved to Israel ""Zionists""?""","""I often hear Pro-Palestinian say things like ""I don't hate Jews, I just hate Zionists."" They explain that ""Zionists"" were a political movement to displace Arabs or something, totally separate from the ethnicity and religion of Judaism. If that were true, then why do they call any Jews who moved to Israel ""Zionists""? For instance, they say things like ""Hundreds of thousands of Zionist colonizers immigrated to Israel in the 1800s and 1900s"" even though the majority of Jews who immigrated to Israel were refugees who had no particular political agenda. There were certainly Jews who dreamed of some kind of vague homeland in Israel — originally the dream was to be Ottoman subjects living in the Ottoman empire in Jewish neighborhoods, later when hundreds of groups started dreaming of a nation state, Jews did too — but the reality is, most Jews moved to escape persecution. For instance, in the 1880s, most Jewish immigrants were Russian and Romanian Jewih refugees fled pogroms (violent anti-…","""Couldn't agree more.""",-1,"""The comment is a one-line expression of agreement ("""
"""oaxrhwt""","""1rw7k69""","""Mamdani rips ‘genocide’ in Palestine at St. Patrick’s Day event — after he botches answer on unified Ireland""","""""","""Mayor Shit-eating-grin""",-1,"""The comment is a short insult/nickname ("""



CONTENT_RELEVANCE_SCORE == 1


comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""nk4k0hx""","""1o9s5u1""","""Venezuelan opposition leader and Nobel Peace Prize winner María Corina Machado voiced support for Israel in a phone call with Netanyahu.""","""""","""She sounds like an Arab leader. I though Latin American elites had a bit more honor than that. At least, ever since the American-supporter Latin death squads went into a decline.""",1,"""The comment makes a vague comparison between a Latin American leader and Arab leaders, but it does not directly address the Gaza conflict or provide relevant insights into public opinion regarding it."""
"""nax85hs""","""1n0bqox""","""Ms Rachel's latest ""wisdom""""","""""","""Jesus is starving in Gaza because the Romans stole most of the food, another thing that inevitably gets blamed on the Jews.""",1,"""The comment references historical events and blames a group for the suffering in Gaza, but it does not engage with the current conflict or provide relevant insights into public opinion on the Gaza situation."""
"""ngoa96l""","""1ns37av""","""Antijudaism, Antisemitism, Antizionism… learn to see it, learn to name it. ✡︎""","""If you don’t know, then you can’t fight. ✡︎ 🇮🇱 ✡︎ The general shape of Jew-hate evolves and morphs over the decades and centuries and millennia to fit more modern sensibilities. But the results are the same: Jew-hate marginalizes and excludes, it destroys Jewish communities, and it puts our People in harm’s way. Fight it every single day. And teach others how to see it and name it and expose it—no matter WHERE it comes from. Shana Tova. ♥️ ""","""All three varieties also include the features of conspiracy theory.""",1,"""The comment discusses conspiracy theories in relation to antisemitism, which is tangentially related to the broader themes of the post but does not directly address the Gaza conflict or its core issues."""
"""nqqyh0x""","""1p6gykk""","""Is Francafrique real or a conspiracy theory? Should the US sanction France for it?""","""Francafrique is a term used to describe France’s sphere of influence in its former colonies. France has intervened militarily in these countries multiple times, and has close economic ties with these countries. Many say the CFA franc grants France tight economic control over these countries and heavily disadvantages them. In recent years rising anti-French sentiment has coincided with the entry of Russian forces into Francafrique. Do you think Francafrique is real and the reason why these countries turned to Russia, or is it an anti-Western conspiracy theory peddled by Russia and other actors? ""","""Don’t we do the same shit with Latin America?""",1,"""The comment draws a parallel between France's influence in its former colonies and US involvement in Latin America, which is a broader ideological discussion not directly tied to the Gaza conflict."""
"""mv5drrq""","""1kzgyse""","""The Gaza Genocide Consensus""","""A hallmark of genocide throughout history is the widespread hesitation to name it while it is still unfolding. The perpetrators and those aligned have clear incentives to deny genocidal intent and actions, even in the face of mounting evidence. This denial serves not only to shield them from legal consequences but also to preserve a collective identity, protect moral self-image, and maintain ideological justifications. At the same time, international actors are reluctant to label ongoing atrocities as genocide, even in the face of mounting evidence, because doing so carries profound moral and legal obligations. Naming genocide compels intervention and confronts deeply entrenched interests. Furthermore, legal processes for determining genocide are slow and cautious, demanding a high standard of proof. In this way, denial does not refute the reality of genocide. Rather, **denial is part of the genocidal process**. However, the current war on Gaza has defied this p


CONTENT_RELEVANCE_SCORE == 2


...

comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""ngpu9mq""","""1nsbe93""","""Professors at Coimbra University are calling to kill Israeli students""","""I had to run away from Portugal after it turned into a lawless land. Europe is gone... That's after our work in the press: https://www.reddit.com/r/Jewish/s/ujG3cfXZPW Justice system does not work, university did not care about calls from the American, Brazilian, German or Israeli embassies after Jews from multiple countries had to escape. The press did not affect them. Seems like we're back in Nazi times. Edit: For those who asked, I know who they are. 1. Yes they are professors (The major one is named at the CAM article, https://combatantisemitism.org/cam-news/one-students-fight-against-antisemitism-in-portugal/ , I've actually taken this picture from her own Instagram account - she's behind as_sentadas). There's also plenty of students but when it's paid faculty it's even more disgusting. 2. Yes they call to murder, they frequently chant for intifada, put pictures of Yahya Sinwar all over campus, and sell stickers claiming Israelis do not have a right to live in the world - …","""They mean Jewish student, no doubt.""",3,"""The comment refers to the targeting of Jewish students and the atmosphere of hostility towards them, which relates to the broader context of antisemitism and violence associated with the Gaza conflict. However, it does not directly discuss the core conflict itself or the situation in Gaza."""
"""ob3r2hj""","""1rwkccm""","""Joseph Kent, who resigned today from his role as Director of Counterterrorism Center (US), believes that Israel manufactured the Syrian Civil War and the spreading of ISIS""","""""","""We don't have the privilege of ending this war soon. If we do, not only will Iran recover, but it will destroy our credibility and leave an entire generation radicalized against us. This war must end with a free Iran, and a peace deal on the white house lawn.""",3,"""The comment discusses the implications of the ongoing war and its impact on regional stability, particularly regarding Iran, which is indirectly related to the Gaza conflict but does not focus on the core issues of the conflict itself."""
"""mz2t17i""","""1lhaiqa""","""Megathread: US President Trump Says That The US Military Has Bombed Multiple Iranian Nuclear Sites""","""At 7:50 p.m. US Eastern, US President Donald Trump posted on Truth Social ""We have completed our very successful attack on the three Nuclear sites in Iran, including Fordow, Natanz, and Esfahan. All planes are now outside of Iran air space. A full payload of BOMBS was dropped on the primary site, Fordow. All planes are safely on their way home. Congratulations to our great American Warriors. There is not another military in the World that could have done this. NOW IS THE TIME FOR PEACE! Thank you for your attention to this matter"". [The AP's live updates page can be found here](https://apnews.com/live/israel-iran-war-updates). --- Trump is set to address the nation; you can find [this subreddit's live discussion thread for that here](https://www.reddit.com/r/politics/comments/1lhc9kv/discussion_thread_us_president_trump_addresses/). --- # Articles that May Interest You | Submission | Domain …","""THE ANTI-WAR PRESIDENT EVERYBODY. IN ONLY 6 FUCKING MONTHS.""",3,"""The comment sarcastically critiques the actions of the US President regarding military strikes, which indirectly relates to public opinion on military involvement in the Gaza conflict and broader Middle Eastern tensions."""
"""nxlhi3i""","""1q32mle""","""Make sure to check out the brilliant Palestine 36 - now available to rent or buy on YouTube!""","""Here’s the link: https://youtu.be/gX49w8SP0CY?si=NLU1F9efjUeKlNWp""","""Appreciate the investment of people in Saudi Arabia in this battle. 👌 🇸🇦""",3,"""The comment mentions Saudi Arabia's investment in the conflict, which re


CONTENT_RELEVANCE_SCORE == 4


comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""nq2f00z""","""1p35ool""","""Sarah Hurwitz, former Obama speechwriter, says the Gaza genocide is imaginary & antisemitic like claims of ""white genocide""/""replacement theory"" in the US.""","""[https://xcancel.com/DropSiteNews/status/1991709907179995401](https://xcancel.com/DropSiteNews/status/1991709907179995401) ""","""Honestly, how would you change people like this? They are either paid to say this and are doing some sick calculation in their heads of using jewish people and the identity as a shield to advance their goals or just genuinely delusional to equate anti racism = anti zionism = antisemitism = facism... wtf.""",4,"""The comment critiques a specific viewpoint on the Gaza conflict and discusses the conflation of anti-racism with anti-Zionism, which relates to public opinion and discourse surrounding the conflict. However, it does not directly address the core events or humanitarian aspects of the situation."""
"""nj9jdcw""","""1o5hbv4""","""Israeli opposition Leader Yair Lapid: Those who demonstrated against Israel in London, Rome, Paris, and Columbia University were deceived by propaganda. Now that the war has stopped, learn the facts: there was no genocide, no intentional starvation in Gaza.” They really hope that we will forget""","""""","""They know that global public opinion of them is totally f***** they're trying to do damage control. It won't work, we have the videos and pictures that their own people posted on their social media bragging about their war crimes.""",4,"""The comment addresses the perception of global public opinion regarding Israel's actions and references war crimes, which are directly related to the ongoing Gaza conflict and its implications."""
"""nm0epol""","""1oj3axz""","""Trump’s unravelling Gaza truce reveals a core truth – neither side wanted it""","""""","""More like Israel didn’t want it the hell are you on Israel keeps breaking it!""",4,"""The comment directly addresses the actions of Israel regarding the Gaza truce, indicating a clear opinion on the conflict and its dynamics, which is relevant to the ongoing situation."""
"""nff6j3r""","""1nmngki""","""If this is the case then you should consider that maybe you are the problem""","""""","""Israel are never asking themselves this. https://preview.redd.it/k8jelyo90jqf1.png?width=711&format=png&auto=webp&s=7c40928ea12937ca9620858e70013eac2f0e4524""",4,"""The comment implies a critique of Israel's perspective on the conflict, which relates to public opinion and the actions of one of the main actors involved in the Gaza conflict."""
"""mm4dd2b""","""1jtig0l""","""What happened to Gazan clans/families to whom we contemplated handing the keys to Gaza?""","""I recall earlier in the war that outlets reported Israel’s desire to hand everyday operations of Gaza to trusted clans/families in the strip. These groups were not Hamas affiliated. Since then, that talk has completely disappeared. The only mention I’ve seen of them lately is that there was a revenge killing by one of these families against a Hamas operative. I’m curious, was this a Gallant-led effort and it has disappeared since he was dismissed from his position? Was it never really a viable option to begin with? Has anyone seen additional information about this? While I’m at it, does anyone have any information about who these clans/families actually are? It’s an interesting alternative solution for post-war Gaza. ""","""From what I've gathered, the clans don't have the kind of power to be able to run it. They're powerful in many ways, but it's not enough.""",4,"""The comment discusses the potential role of Gazan clans in the governance of Gaza and their relationship with Hamas, which directly relates to the dynamics of the conflict and post-war considerations."""



CONTENT_RELEVANCE_SCORE == 5


comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""o05l3lw""","""1qfa6m3""","""White House announced the Gaza ""board of peace"" of zionists and war criminals, with no palestinian representation.""","""""","""I hope that Hamas will keep ruling the Gaza Strip, will keep resisting against the abusive genocidal oppressive sadistic childkilling genocidal maniacs and Zionist occupiers and that they will be able to represent the Palestinian people in the best way possible. I wish only curses and bad things in life for all those that are plotting against the Palestinians to help the Zionist Entity in any form possible.""",5,"""The comment explicitly discusses Hamas's role in Gaza and expresses strong opinions about the conflict, including references to the actions of Zionists and the plight of Palestinians, which are directly related to the core conflict."""
"""nbg8r8m""","""1n3tjm5""","""The argument for why Oct. 7 meets the definition of genocide committed by Hamas against Israeli Jews""","""Avraham Shalev’s article advances the argument that the Hamas assault of October 7, 2023, constitutes genocide under international law. He builds his case by applying the two essential elements of genocide found in the 1948 Genocide Convention—the physical element (actus reus) and the mental element (dolus specialis)—and by highlighting the novel political tactic he calls “genocide inversion,” whereby perpetrators accuse their victims of the very crime they themselves have committed. On the physical side, the atrocities carried out on October 7 were wide-ranging, systematic, and explicitly directed against civilians. Coordinated attacks on 22 Israeli towns and the Nova music festival left more than 1,200 civilians dead and thousands more wounded. Survivors and investigators documented torture, mutilation, abductions, and systematic sexual violence, including gang rapes and the desecration of bodies. These acts fall squarely under the prohibitions of Article II of the Convention, which…","""Eyyyy I was considering making this post down the road. Glad someone did it for me. While I think personally (not legally) that its debatable that October 7th itself can be fully classified as a genocide, I think its beyond dispute that Hamas had genocidal intent, definitely more compelling than what we see with Israel. That coupled with their crimes against humanity would seem to meet the threshold for genocide.""",5,"""The comment engages directly with the topic of genocide in relation to the October 7 attacks, discussing both the intent of Hamas and the classification of the event, which are central to the Gaza conflict."""
"""ngc48y6""","""1nr5tx7""","""israelis shoot a palestinian teen after they tell him to walk away.""","""""","""Fuck IDF terrorists!""",5,"""The comment expresses a strong opinion about the IDF in the context of violence against a Palestinian teen, directly relating to the core conflict and events occurring in Gaza."""
"""niszq4p""","""1o2r9ga""","""Your silence is deafening""","""All supposedly pro-Palestinian activists who are silent on the proposed peace deal which would provide the short term ceasefire you claim you wanted as well as, prevention of any annexation of Gaza and bring in massive aid into Gaza in the short term and long term - your silence is deafening. This deal is not perfect but it giving Palestinians of Gaza everything you said you wanted - so why do not vocally support it? That’s because it’s also good for Israel and bad for Hamas. You have been so busy deluding yourself into thinking the enemy of your enemy is your friend you have doubled and tripled down on Hamas being ‘freedom fighters’. Yet Hamas has been the most proximate cause of suffering in Gaza since 2006. Hamas is Not what’s best for Palestinians. Hamas has used the suffering of Palestinians to persuade so many of you that Israel is the bad guy and they (by being an

In [ ]:
processed_df.filter(pl.col('content_relevance_score') < 3)

comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""n11hxav""","""1lloezh""","""How can anyone still deny it's a genocide in Gaza?""","""New Haaretz investigation: Israeli soldiers have provided shocking testimonials of the state of operations in Gaza. One IDF solider said: > We open fire early in the morning if someone tries to get in line from a few hundred meters away, and sometimes we just charge at them from close range. But there's no danger to the forces. I'm not aware of a single instance of return fire. There's no enemy, no weapons. Another described the area near the food trucks as “a killing field.” > Where I was stationed, between one and five people were killed every day. They're treated like a hostile force – no crowd-control measures, no tear gas – just live fire with everything imaginable: heavy machine guns, grenade launchers, mortars. Then, once the center opens, the shooting stops, and they know they can approach. Our form of communication is gunfire. Yet another soldier stated > Technically, it's supposed to be warning fire – either to push people back or stop them from advancing,"" he said. ""But…","""Because it isn't one, a genocide is when a government or group is purposely setting out to kill everyone indiscriminately like the holocaust, this is just war and in war there will always be deaths of civilians. This is unfortunate but anyone with a logical brain knows israel could have done far worse to gaza if they were truly hell bent on evil, they have the firepower to erase the gaza strip in a couple days easily.""",2,"""The comment dismisses the characterization of the situation in Gaza as genocide, framing it instead as a consequence of war, which reflects a specific opinion on the conflict. However, it does not engage with the core humanitarian issues or the direct actions described in the original post, making it less relevant."""
"""mu58xtn""","""1ku8sal""","""No words""","""""","""Even the stats on it are fake . More comments than likes 🤣🤣🤣""",0,"""The comment does not address the Gaza conflict or any related topics, focusing instead on social media engagement metrics, which is irrelevant to the study."""
"""n61xsyu""","""1mdi98j""","""Jon Stewart called Israel a failure of humanity. Beinart let It slide. An opportunity missed.""","""I watched Peter Beinart on Jon Stewart, and while I respect parts of his moral appeal, his framing reveals a dangerous naivety. Especially for those of us who’ve worked in developing countries, engaged with the Palestinian movement, and understand what Zionism actually responds to. **Full thoughts below.** Jon Stewart said Israel’s existence is a “failure of humanity.” And you know what? He might be right - but not how he means it. The failure isn’t that Israel exists. It’s that it *has to*. After millennia of persecution, pogroms, ghettos, ethnic cleansing, and global apathy toward our slaughter... Jews had no choice but to build a fortress in the desert. And now that we have, the world demands we apologize for it. Beinart speaks about humanizing Palestinians. **Good**. We should. But he doesn’t acknowledge the failures of Palestinian leadership, or of a movement that has chosen grievance over growth time and again, while amplifying the rare nonviolent moments as if they defin…","""Let me guess John Stewart is Hamas Propaganda""",2,"""The comment dismisses Jon Stewart's views by labeling them as Hamas propaganda, which reflects a polarized opinion on the Gaza conflict but does not engage with the core issues or provide substantial insight into the conflict itself."""
"""nndulce""","""1opqxl5""","""“I can no longer protect New Yorkers from fires because the Mayor criticized the foreign country of Israel”""","""""","""Great! He can fuck off then, replace him with someone capable now""",1,"""The comment focuses on a domestic political reaction to a mayor's criticism of Israel, which